In [8]:
import jax
import jax.numpy as jnp
import chex


# MOderately complicated physics formula.
def f(inp):
    out0 = jnp.log(jnp.sin(inp["x"]) + jnp.cos(inp["pie"]))
    out1 = jnp.sqrt(jnp.abs(jnp.cos(jnp.sin(inp["x"] * inp["pie"]) * jnp.exp(inp["x"]))))
    out = {"foo": out0, "bar": out1}
    return out


jac_f = jax.jacfwd(f)

# Evaluate the Jacobian at a point.
inp = {"x": 1.0, "pie": 2.0}
value = f(inp)
jacobian = jac_f(inp)
print(value)
print(jacobian)

{'foo': Array(-0.8549038, dtype=float32, weak_type=True), 'bar': Array(0.885384, dtype=float32, weak_type=True)}
{'bar': {'pie': Array(-0.396632, dtype=float32, weak_type=True), 'x': Array(0.07339273, dtype=float32, weak_type=True)}, 'foo': {'pie': Array(-2.137893, dtype=float32, weak_type=True), 'x': Array(1.2703307, dtype=float32, weak_type=True)}}


In [9]:
%load_ext autoreload
%autoreload 2

from pprint import pprint

import popsim.param_utils as param_utils
from popsim.simulators.comet_mirror.scenarios.sparc_prd import build_comet_mirror_config

# Initialize the simulator.
model, state, params = build_comet_mirror_config()

# Make a time base for all of our simulations.
time_base = param_utils.make_time_base(t0=0.0, t1=5.0, dt=0.01)

from copy import deepcopy

import jax.numpy as jnp

from popsim.enums import Impurity
from popsim.simulators.comet_mirror.simulate import simulate, build_vectorized_params, _vec_simulate

# Create a copy of "params". It's best to keep the original one around untouched.
new_params = deepcopy(params)

# Manually define a current-ramp where the key is the time in seconds and the value is the current in Amperes.
new_params.plasma_current = {0.0: 8.7e6, 1.0: 8.7e6, 5.0: 4.0e6}

# Manually define an auxiliary heating power ramp where the key is the time in seconds and the value is the power in MW.
new_params.P_aux_MW = {0.0: 11.1, 5.0: 7.0}

# Manually define a quick tungsten spike. Note that under the hood linear interpolation is happening, so we need this
# perhaps somewhat awkward definition.
new_params.fueling19[Impurity.Tungsten] = {
    0.0: 0.0,
    1.99: 0.0,  # Start ramping impurities.
    2.0: 0.1,  # Impurity injection.
    2.1: 0.1,  # Impurity injection holding.
    2.11: 0.0,  # Impurity drops back to 0.0.
    5.0: 0.0,  # Impurity holds at 0.0.
}

params_vectorized, multi_sim = build_vectorized_params(params, time_base, "linear")


def sim_with_params(params):
    return _vec_simulate(model, ts=time_base, state=state, params=params)


sim_with_params(params_vectorized)

TypeError: Value 'magnetic_field_on_axis' with dtype <U22 is not a valid JAX array type. Only arrays of numeric types are supported by JAX.